In [ ]:
#pip install imageio

In [ ]:
# PCA
# 画出解释方差曲线, 找到合适的保留大部分信息的拐点, 然后将这个点作为输入数据的341维度的目标降维维度

# 数据分割\

# 训练标签保存要方便加载

# 训练后也加入AUCPR

# 数据集划分验证

In [ ]:
# 导入必要的库
import os
import time
from fastkan import *
from fastkan import FastKAN
import random
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import matplotlib.pyplot as plt
import matplotlib.patches as mpts
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, cohen_kappa_score, accuracy_score
from sklearn.metrics import precision_score,precision_recall_curve, auc, recall_score, f1_score, accuracy_score, confusion_matrix

from sklearn.preprocessing import minmax_scale
import pandas as pd
from scipy.io import loadmat
from tqdm.notebook import tqdm
from IPython import display
import h5py
import copy
import sys
import glob
import seaborn as sns
from datetime import datetime
%matplotlib inline

In [ ]:
## 超参数和实验设置配置单元格

# 设置随机种子，确保实验可重复性
RANDOM_SEED = 666

# 数据预处理参数
APPLY_PCA = True   # 是否应用PCA降维
N_PCA = 0          # 设为0表示自动选择主成分数量，大于0表示使用指定数量
NORM = True        # 是否对数据进行标准化/归一化处理

# 定义模型名称，用于结果保存和模型标识
MODEL_NAME = 'BrainVoxel_1DKAN'

# 指定数据集名称
DATASET = 'BrainVoxel'

# 数据集划分比例
TRAIN_RATE = 0.7  # 将原训练集按7:3分割，70%用于训练
TEST_RATE = 0.3   # 30%用于测试
# 原验证集保持不变，用作最终验证

# 训练参数
EPOCH = 100        # 总训练轮数
VAL_EPOCH = 1      # 每隔多少轮进行一次验证
LR = 0.001         # 学习率
WEIGHT_DECAY = 1e-6  # 权重衰减系数，用于L2正则化
BATCH_SIZE = 64    # 批处理大小，固定不变

# 计算设备选择
DEVICE = 0         # -1表示使用CPU，0表示使用第一块GPU(cuda:0)

# 数据参数
FEATURE_DIM = 341  # 输入特征维度
NUM_CLASS = 2      # 二分类问题：正类和负类
FIXED_GRID = 3     # 固定网格大小，不进行网格扩展


# 模型检查点路径
CHECK_POINT = None  # 加载预训练模型的路径，None表示从头开始训练

# 结果保存路径
SAVE_PATH = f"./Results/{MODEL_NAME}/{DATASET}"
# 如果保存目录不存在，则创建该目录
if not os.path.isdir(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [ ]:
## 设置随机数种子，确保实验结果可复现

# 为Python的random模块设置随机种子
random.seed(RANDOM_SEED)

# 为PyTorch的CPU操作设置随机种子
torch.manual_seed(RANDOM_SEED)

# 为当前GPU设置随机种子
torch.cuda.manual_seed(RANDOM_SEED)

# 为所有可用GPU设置相同的随机种子
torch.cuda.manual_seed_all(RANDOM_SEED)

# 为NumPy库设置随机种子
np.random.seed(RANDOM_SEED)

# 禁用CuDNN的非确定性算法
torch.backends.cudnn.deterministic = True

# 禁用CuDNN的自动优化选择
torch.backends.cudnn.benchmark = False

In [ ]:
# 脑体素数据载入工具函数
def load_brain_voxel_data():
    """
    载入脑体素数据、标签和训练/验证/测试集
    
    返回:
        data: 高维特征数据
        train_gt: 训练集标签
        val_gt: 验证集标签
        all_data_dict: 包含所有数据集信息的字典
    """
    # 路径配置 - 根据您的实际路径进行调整
    output_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output'
    train_label_dir = os.path.join(output_path, 'train_set_by_label')
    val_label_dir = os.path.join(output_path, 'val_set_by_label')
    
    # 读取标签索引文件
    def load_label_index(index_file):
        label_info = {}
        with open(index_file, 'r') as f:
            # 跳过表头
            next(f)
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3:
                    label_id = int(parts[0])
                    voxel_count = int(parts[1])
                    filename = parts[2] if parts[2] else None
                    label_info[label_id] = {'count': voxel_count, 'filename': filename}
        return label_info
    
    train_index_file = os.path.join(train_label_dir, "label_index.txt")
    val_index_file = os.path.join(val_label_dir, "val_label_index.txt")
    
    if not os.path.exists(train_index_file):
        raise FileNotFoundError(f"训练标签索引文件不存在: {train_index_file}")
    if not os.path.exists(val_index_file):
        raise FileNotFoundError(f"验证标签索引文件不存在: {val_index_file}")
    
    train_label_info = load_label_index(train_index_file)
    val_label_info = load_label_index(val_index_file)
    
    # 获取有效标签（有体素数据的标签）
    valid_labels = [label_id for label_id, info in train_label_info.items() 
                   if info['count'] > 0]
    
    print(f"数据加载完成: 找到 {len(valid_labels)} 个有效标签")
    
    return {
        'train_label_info': train_label_info,
        'val_label_info': val_label_info,
        'valid_labels': valid_labels,
        'train_label_dir': train_label_dir,
        'val_label_dir': val_label_dir
    }

# 加载数据集信息
all_data_dict = load_brain_voxel_data()

In [ ]:
def apply_pca(X, num_components=15, norm=True):
    """
    对数据进行PCA降维和标准化处理
    
    参数:
        X (ndarray): 需要降维的数据
        num_components (int): 保留的主成分数量，0表示不进行PCA
        norm (bool): 是否进行标准化处理
    
    返回:
        new_X: 处理后的数据
        num_components: 最终的特征维度
    """
    if num_components == 0:
        # 不进行PCA，但可能进行标准化
        if norm:
            # 对每个特征进行标准化
            mean = np.mean(X, axis=0)
            std = np.std(X, axis=0)
            # 避免除以0
            std[std == 0] = 1
            new_X = (X - mean) / std
        else:
            new_X = X.copy()
        return new_X, X.shape[1]
    else:
        # 进行PCA降维
        pca = PCA(n_components=num_components)
        new_X = pca.fit_transform(X)
        
        # 可选的标准化
        if norm:
            # 对PCA后的特征进行归一化
            new_X = (new_X - np.min(new_X, axis=0)) / (np.max(new_X, axis=0) - np.min(new_X, axis=0) + 1e-10)
        
        return new_X, new_X.shape[1]

def analyze_pca_variance(X, max_components=None, plot=True, save_path=None):
    """
    分析PCA的方差解释率，找到合适的降维维度
    
    参数:
        X (ndarray): 输入数据
        max_components (int): 最大考虑的主成分数，None表示使用特征维度
        plot (bool): 是否绘制解释方差曲线
        save_path (str): 保存图像的路径，None表示不保存
        
    返回:
        optimal_n_components: 建议的主成分数量
    """
    # 确定最大主成分数
    if max_components is None:
        max_components = min(X.shape[0], X.shape[1])
    else:
        max_components = min(max_components, X.shape[0], X.shape[1])
    
    # 计算所有可能的主成分
    pca = PCA(n_components=max_components)
    pca.fit(X)
    
    # 计算累积解释方差
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
    
    # 寻找方差解释率达到95%的拐点
    threshold = 0.95
    optimal_n_components = np.argmax(cumulative_variance_ratio >= threshold) + 1
    
    # 寻找拐点（斜率变化最大的点）
    gradient = np.gradient(explained_variance_ratio)
    gradient_of_gradient = np.gradient(gradient)
    elbow_index = np.argmax(np.abs(gradient_of_gradient))
    elbow_n_components = elbow_index + 1
    
    if plot:
        plt.figure(figsize=(12, 6))
        
        # 绘制方差解释率
        plt.subplot(1, 2, 1)
        plt.plot(range(1, len(explained_variance_ratio) + 1), 
                 explained_variance_ratio, 'bo-', markersize=4)
        plt.axvline(x=elbow_n_components, color='r', linestyle='--', 
                    label=f'拐点: {elbow_n_components}维')
        plt.xlabel('主成分数量')
        plt.ylabel('解释方差比例')
        plt.title('各主成分解释方差比例')
        plt.grid(True)
        plt.legend()
        
        # 绘制累积方差
        plt.subplot(1, 2, 2)
        plt.plot(range(1, len(cumulative_variance_ratio) + 1), 
                 cumulative_variance_ratio, 'ro-', markersize=4)
        plt.axhline(y=threshold, color='g', linestyle='--', 
                    label=f'{threshold*100}%方差')
        plt.axvline(x=optimal_n_components, color='b', linestyle='--', 
                    label=f'阈值维度: {optimal_n_components}')
        plt.xlabel('主成分数量')
        plt.ylabel('累积解释方差比例')
        plt.title('累积解释方差比例')
        plt.grid(True)
        plt.legend()
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path)
        plt.show()
    
    print(f"方差拐点对应的主成分数量: {elbow_n_components}")
    print(f"达到{threshold*100}%方差解释率需要的主成分数量: {optimal_n_components}")
    print(f"前{optimal_n_components}个主成分解释了总方差的{cumulative_variance_ratio[optimal_n_components-1]*100:.2f}%")
    
    # 返回拐点和阈值点的较小值作为建议
    suggested_components = min(elbow_n_components, optimal_n_components)
    return suggested_components, explained_variance_ratio, cumulative_variance_ratio


In [ ]:
class BrainVoxelDataset(Dataset):
    """
    脑体素数据集类，简化版
    """
    def __init__(self, data, labels, is_inference=False):
        """
        初始化数据集
        
        参数:
            data: 特征数据，形状为(n_samples, feature_dim)
            labels: 标签数据，形状为(n_samples,)
            is_inference: 是否为推理模式（不返回标签）
        """
        super(BrainVoxelDataset, self).__init__()
        self.data = data
        self.labels = labels
        self.is_inference = is_inference
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx]
        x = torch.FloatTensor(x)
        
        if self.is_inference:
            return x
        else:
            y = self.labels[idx]
            y = torch.LongTensor([int(y)])[0]  # 显式转换为整数
            return x, y

In [ ]:
def get_dataset_for_label(label_id, all_data_dict, train_valid_test_ratio=[0.6, 0.2, 0.2], 
                         apply_pca_flag=True, n_components=15, norm=True):
    """
    获取指定标签的数据集，并按指定比例分割为训练集、验证集和测试集
    
    参数:
        label_id: 目标标签ID
        all_data_dict: 包含数据路径和信息的字典
        train_valid_test_ratio: 训练集、验证集、测试集的比例，默认[0.6, 0.2, 0.2]
        apply_pca_flag: 是否应用PCA
        n_components: PCA保留的主成分数量，0表示自动选择
        norm: 是否进行标准化处理
    
    返回:
        dataset_dict: 包含训练集、验证集和测试集的字典
    """
    train_label_dir = all_data_dict['train_label_dir']
    val_label_dir = all_data_dict['val_label_dir']
    valid_labels = all_data_dict['valid_labels']
    train_label_info = all_data_dict['train_label_info']
    val_label_info = all_data_dict['val_label_info']
    
    if label_id not in valid_labels:
        raise ValueError(f"标签 {label_id} 不在有效标签列表中")
    
    # 获取目标标签的文件路径
    def get_label_file_path(label_id, is_validation=False):
        if is_validation:
            pattern = os.path.join(val_label_dir, f"label_{label_id}_count_*_voxels.npy")
        else:
            pattern = os.path.join(train_label_dir, f"label_{label_id}_count_*_voxels.npy")
        
        matches = glob.glob(pattern)
        return matches[0] if matches else None
    
    # 加载正样本（目标标签的体素）
    train_file = get_label_file_path(label_id, False)
    val_file = get_label_file_path(label_id, True)
    
    if not train_file:
        raise ValueError(f"标签 {label_id} 没有训练数据文件")
    
    print(f"标签 {label_id} 信息:")
    print(f"  训练集体素数: {train_label_info[label_id]['count'] if label_id in train_label_info else 0}")
    print(f"  验证集体素数: {val_label_info[label_id]['count'] if label_id in val_label_info else 0}")
    
    # 加载训练数据（正样本）
    train_positive_samples = np.load(train_file)
    train_positive_labels = np.ones(len(train_positive_samples))
    
    # 加载验证数据（正样本）
    val_positive_samples = np.load(val_file) if val_file else np.array([])
    val_positive_labels = np.ones(len(val_positive_samples)) if val_file else np.array([])
    
    # 合并训练和验证集的正样本
    all_positive_samples = np.vstack([train_positive_samples, val_positive_samples]) if len(val_positive_samples) > 0 else train_positive_samples
    all_positive_labels = np.concatenate([train_positive_labels, val_positive_labels]) if len(val_positive_labels) > 0 else train_positive_labels
    
    # 获取负样本（来自其他标签）
    other_labels = [l for l in valid_labels if l != label_id]
    random.shuffle(other_labels)
    
    # 保持与正样本相同数量的负样本
    target_negative_count = len(all_positive_samples)
    train_negative_samples = []
    val_negative_samples = []
    current_train_count = 0
    current_val_count = 0
    
    for other_label in other_labels:
        # 如果已经收集足够的负样本，则停止
        if current_train_count + current_val_count >= target_negative_count:
            break
            
        # 加载训练集的负样本
        other_train_file = get_label_file_path(other_label, False)
        if other_train_file and current_train_count < target_negative_count // 2:
            other_train_samples = np.load(other_train_file)
            samples_to_take = min(len(other_train_samples), 
                                 target_negative_count // 2 - current_train_count)
            
            if samples_to_take < len(other_train_samples):
                # 随机抽取部分样本
                indices = np.random.choice(len(other_train_samples), samples_to_take, replace=False)
                selected_samples = other_train_samples[indices]
            else:
                selected_samples = other_train_samples
            
            train_negative_samples.append(selected_samples)
            current_train_count += len(selected_samples)
        
        # 加载验证集的负样本
        other_val_file = get_label_file_path(other_label, True)
        if other_val_file and current_val_count < target_negative_count // 2:
            other_val_samples = np.load(other_val_file)
            samples_to_take = min(len(other_val_samples), 
                                 target_negative_count // 2 - current_val_count)
            
            if samples_to_take < len(other_val_samples):
                # 随机抽取部分样本
                indices = np.random.choice(len(other_val_samples), samples_to_take, replace=False)
                selected_samples = other_val_samples[indices]
            else:
                selected_samples = other_val_samples
            
            val_negative_samples.append(selected_samples)
            current_val_count += len(selected_samples)
    
    # 合并负样本
    all_train_negative = np.vstack(train_negative_samples) if train_negative_samples else np.array([]).reshape(0, train_positive_samples.shape[1])
    all_val_negative = np.vstack(val_negative_samples) if val_negative_samples else np.array([]).reshape(0, train_positive_samples.shape[1])
    
    # 合并训练和验证集的负样本
    all_negative_samples = np.vstack([all_train_negative, all_val_negative]) if len(all_train_negative) > 0 and len(all_val_negative) > 0 else (all_train_negative if len(all_train_negative) > 0 else all_val_negative)
    all_negative_labels = np.zeros(len(all_negative_samples))
    
    # 确保负样本数量与正样本数量相同
    if len(all_negative_samples) > target_negative_count:
        indices = np.random.choice(len(all_negative_samples), target_negative_count, replace=False)
        all_negative_samples = all_negative_samples[indices]
        all_negative_labels = all_negative_labels[indices]
    
    # 如果进行PCA，我们使用所有数据进行拟合
    if apply_pca_flag:
        print("应用PCA降维...")
        
        # 合并所有样本进行PCA拟合（不包括标签）
        all_samples_for_pca = np.vstack([all_positive_samples, all_negative_samples])
        
        # 分析PCA方差并获取建议的主成分数量
        if n_components == 0:  # 如果用户没有指定主成分数量，则自动分析
            suggested_components, _, _ = analyze_pca_variance(
                all_samples_for_pca, 
                plot=True,
                save_path=os.path.join(SAVE_PATH, f'pca_variance_analysis_label_{label_id}.png')
            )
            n_components = suggested_components
            print(f"自动选择了{n_components}个主成分进行降维")
        
        # 应用PCA到所有数据
        all_samples_pca, feature_dim = apply_pca(all_samples_for_pca, n_components, norm)
        
        # 分割回正样本和负样本
        all_positive_samples = all_samples_pca[:len(all_positive_samples)]
        all_negative_samples = all_samples_pca[len(all_positive_samples):]
    else:
        print("不应用PCA，使用原始特征...")
        feature_dim = all_positive_samples.shape[1]
    
    # 分别对正样本和负样本进行划分，确保类别平衡
    # 正样本划分
    positive_indices = np.arange(len(all_positive_samples))
    np.random.shuffle(positive_indices)
    
    train_end_pos = int(len(positive_indices) * train_valid_test_ratio[0])
    valid_end_pos = train_end_pos + int(len(positive_indices) * train_valid_test_ratio[1])
    
    train_pos_indices = positive_indices[:train_end_pos]
    valid_pos_indices = positive_indices[train_end_pos:valid_end_pos]
    test_pos_indices = positive_indices[valid_end_pos:]
    
    # 负样本划分
    negative_indices = np.arange(len(all_negative_samples))
    np.random.shuffle(negative_indices)
    
    train_end_neg = int(len(negative_indices) * train_valid_test_ratio[0])
    valid_end_neg = train_end_neg + int(len(negative_indices) * train_valid_test_ratio[1])
    
    train_neg_indices = negative_indices[:train_end_neg]
    valid_neg_indices = negative_indices[train_end_neg:valid_end_neg]
    test_neg_indices = negative_indices[valid_end_neg:]
    
    # 组合训练集
    train_samples = np.vstack([
        all_positive_samples[train_pos_indices],
        all_negative_samples[train_neg_indices]
    ])
    train_labels = np.concatenate([
        np.ones(len(train_pos_indices)),
        np.zeros(len(train_neg_indices))
    ])
    
    # 组合验证集
    valid_samples = np.vstack([
        all_positive_samples[valid_pos_indices],
        all_negative_samples[valid_neg_indices]
    ])
    valid_labels = np.concatenate([
        np.ones(len(valid_pos_indices)),
        np.zeros(len(valid_neg_indices))
    ])
    
    # 组合测试集
    test_samples = np.vstack([
        all_positive_samples[test_pos_indices],
        all_negative_samples[test_neg_indices]
    ])
    test_labels = np.concatenate([
        np.ones(len(test_pos_indices)),
        np.zeros(len(test_neg_indices))
    ])
    
    # 随机打乱每个集合
    def shuffle_data(samples, labels):
        indices = np.arange(len(samples))
        np.random.shuffle(indices)
        return samples[indices], labels[indices]
    
    train_samples, train_labels = shuffle_data(train_samples, train_labels)
    valid_samples, valid_labels = shuffle_data(valid_samples, valid_labels)
    test_samples, test_labels = shuffle_data(test_samples, test_labels)
    
    print(f"数据集创建完成:")
    print(f"  总样本数: {len(all_positive_samples) + len(all_negative_samples)} (正样本: {len(all_positive_samples)}, 负样本: {len(all_negative_samples)})")
    print(f"  训练集: {len(train_samples)} 样本 (正样本: {np.sum(train_labels==1)}, 负样本: {np.sum(train_labels==0)})")
    print(f"  验证集: {len(valid_samples)} 样本 (正样本: {np.sum(valid_labels==1)}, 负样本: {np.sum(valid_labels==0)})")
    print(f"  测试集: {len(test_samples)} 样本 (正样本: {np.sum(test_labels==1)}, 负样本: {np.sum(test_labels==0)})")
    print(f"  特征维度: {feature_dim}")
    
    # 计算各集合中的正负样本比例
    train_pos_ratio = np.mean(train_labels)
    valid_pos_ratio = np.mean(valid_labels)
    test_pos_ratio = np.mean(test_labels)
    
    print(f"  训练集正样本比例: {train_pos_ratio*100:.1f}%")
    print(f"  验证集正样本比例: {valid_pos_ratio*100:.1f}%")
    print(f"  测试集正样本比例: {test_pos_ratio*100:.1f}%")
    
    return {
        'train_samples': train_samples,
        'train_labels': train_labels,
        'val_samples': valid_samples, 
        'val_labels': valid_labels,
        'test_samples': test_samples,
        'test_labels': test_labels,
        'feature_dim': feature_dim
    }


def prepare_all_labels_datasets(all_data_dict, target_labels=None, 
                              train_valid_test_ratio=[0.6, 0.2, 0.2],
                              apply_pca_flag=True, n_components=15, norm=True):
    """
    准备所有标签或指定标签的数据集
    
    参数:
        all_data_dict: 数据字典
        target_labels: 要处理的标签列表，None表示处理所有标签
        apply_pca_flag: 是否应用PCA
        n_components: PCA保留的主成分数
        norm: 是否标准化数据
    
    返回:
        datasets_dict: 包含每个标签数据集的字典
    """
    if target_labels is None:
        target_labels = all_data_dict['valid_labels']
    
    datasets_dict = {}
    
    for label_id in tqdm(target_labels, desc="处理标签"):
        try:
            dataset = get_dataset_for_label(
                label_id, 
                all_data_dict,
                train_valid_test_ratio=[TRAIN_RATE, (1-TRAIN_RATE)/2, (1-TRAIN_RATE)/2],  # 添加这个参数
                apply_pca_flag=apply_pca_flag,
                n_components=n_components,
                norm=norm
            )
            datasets_dict[label_id] = dataset
            
            # 打印进度信息
            print(f"标签 {label_id} 处理完成: "
                  f"训练集 {len(dataset['train_samples'])} 样本, "
                  f"验证集 {len(dataset['val_samples'])} 样本, "
                  f"测试集 {len(dataset['test_samples'])} 样本")
            
        except Exception as e:
            print(f"处理标签 {label_id} 时出错: {str(e)}")
    
    return datasets_dict

# 在prepare_all_labels_datasets函数之后添加
def validate_datasets(datasets_dict):
    """
    验证处理后的数据集
    
    参数:
        datasets_dict: 包含每个标签数据集的字典
    """
    for label_id, dataset in datasets_dict.items():
        train_pos = np.sum(dataset['train_labels'] == 1)
        train_neg = np.sum(dataset['train_labels'] == 0)
        val_pos = np.sum(dataset['val_labels'] == 1)
        val_neg = np.sum(dataset['val_labels'] == 0)
        test_pos = np.sum(dataset['test_labels'] == 1)
        test_neg = np.sum(dataset['test_labels'] == 0)
        
        print(f"标签 {label_id} 验证:")
        print(f"  训练集: 正样本 {train_pos} ({train_pos/(train_pos+train_neg)*100:.1f}%), "
              f"负样本 {train_neg} ({train_neg/(train_pos+train_neg)*100:.1f}%)")
        print(f"  验证集: 正样本 {val_pos} ({val_pos/(val_pos+val_neg)*100:.1f}%), "
              f"负样本 {val_neg} ({val_neg/(val_pos+val_neg)*100:.1f}%)")
        print(f"  测试集: 正样本 {test_pos} ({test_pos/(test_pos+test_neg)*100:.1f}%), "
              f"负样本 {test_neg} ({test_neg/(test_pos+test_neg)*100:.1f}%)")

In [ ]:

class BrainVoxelKAN(nn.Module):
    """
    用于脑体素分类的KAN模型，简化版
    """
    def __init__(self, input_dim, hidden_dim, num_classes, grid_size=3):
        """
        初始化模型
        
        参数:
            input_dim: 输入特征维度
            hidden_dim: 隐藏层维度
            num_classes: 类别数量
            grid_size: 网格大小，保持固定
        """
        super(BrainVoxelKAN, self).__init__()
        
        self.kan = FastKAN(
            layers_hidden=[input_dim, hidden_dim, num_classes],
            num_grids=grid_size
        )
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征，形状为(batch_size, input_dim)
            
        返回:
            output: 模型输出，形状为(batch_size, num_classes)
        """
        return self.kan(x)

In [ ]:
def train_brain_voxel_kan(model, train_loader, test_loader, criterion, optimizer, device, 
                          num_epochs=100, val_epoch=1, save_path="./Results"):
    """
    训练脑体素KAN模型，与1D KAN训练流程保持一致，并增加更全面的指标评估
    
    参数:
        model: KAN模型
        train_loader: 训练数据加载器
        test_loader: 测试数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        num_epochs: 训练轮数
        val_epoch: 验证频率
        save_path: 模型保存路径
    
    返回:
        训练结果统计信息
    """
    # 初始化统计变量
    loss_list = []
    acc_list = []
    val_acc_list = []
    val_epoch_list = []
    val_f1_list = []
    val_auc_pr_list = []
    val_recall_list = []
    
    # 保存起始时间
    train_st = time.time()
    
    # 计算批次数量和样本数量
    batch_num = len(train_loader)
    train_num = len(train_loader.dataset)
    test_num = len(test_loader.dataset)
    
    try:
        # 训练循环
        for e in tqdm(range(num_epochs), desc="Training:"):
            # 设置模型为训练模式
            model.train()
            avg_loss = 0.0
            train_acc = 0
            
            # 批次循环
            for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=batch_num):
                # 将数据移动到指定设备
                data, target = data.to(device), target.to(device)
                
                # 前向传播
                optimizer.zero_grad()
                out = model(data)
                loss = criterion(out, target)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                # 累计损失和准确率
                avg_loss += loss.item()
                _, pred = torch.max(out, dim=1)
                train_acc += (pred == target).sum().item()
            
            # 计算本轮平均损失和准确率
            loss_list.append(avg_loss / train_num)
            acc_list.append(train_acc / train_num)
            print(f"epoch {e}/{num_epochs} loss:{loss_list[-1]}  acc:{acc_list[-1]}")
            
            # 验证阶段
            if (e+1) % val_epoch == 0 or (e+1) == num_epochs:
                val_acc = 0
                model.eval()
                
                # 收集验证数据的预测结果
                all_preds = []
                all_probs = []
                all_targets = []
                
                with torch.no_grad():
                    for batch_idx, (data, target) in tqdm(enumerate(test_loader), total=len(test_loader)):
                        data, target = data.to(device), target.to(device)
                        out = model(data)
                        probs = torch.softmax(out, dim=1)
                        _, pred = torch.max(out, dim=1)
                        
                        all_preds.extend(pred.cpu().numpy())
                        all_probs.extend(probs[:, 1].cpu().numpy())  # 保存正类概率
                        all_targets.extend(target.cpu().numpy())
                        val_acc += (pred == target).sum().item()
                
                # 计算全面的评估指标
                val_accuracy = val_acc / test_num
                val_recall = recall_score(all_targets, all_preds, average='binary', zero_division=0)
                val_precision = precision_score(all_targets, all_preds, average='binary', zero_division=0)
                val_f1 = f1_score(all_targets, all_preds, average='binary', zero_division=0)
                
                # 计算AUC-PR
                precision_curve, recall_curve, _ = precision_recall_curve(all_targets, all_probs)
                val_auc_pr = auc(recall_curve, precision_curve)
                
                # 保存验证结果
                val_acc_list.append(val_accuracy)
                val_epoch_list.append(e)
                val_f1_list.append(val_f1)
                val_auc_pr_list.append(val_auc_pr)
                val_recall_list.append(val_recall)
                
                # 显示全面的评估指标
                print(f"epoch {e}/{num_epochs}  val_acc:{val_accuracy:.4f}  val_f1:{val_f1:.4f}  val_recall:{val_recall:.4f}  val_auc_pr:{val_auc_pr:.4f}")
                
                # 打印混淆矩阵
                conf_matrix = confusion_matrix(all_targets, all_preds)
                print(f"Confusion Matrix:\n{conf_matrix}")
                print(f"True Positives: {conf_matrix[1][1]}, False Positives: {conf_matrix[0][1]}")
                print(f"True Negatives: {conf_matrix[0][0]}, False Negatives: {conf_matrix[1][0]}")
                
                # 保存当前模型
                save_name = os.path.join(save_path, f"epoch_{e}_acc_{val_accuracy:.4f}_f1_{val_f1:.4f}_aucpr_{val_auc_pr:.4f}.pth")
                save_dict = {
                    'state_dict': model.state_dict(), 
                    'epoch': e+1, 
                    'optimizer': optimizer.state_dict(),
                    'loss_list': loss_list, 
                    'acc_list': acc_list, 
                    'val_acc_list': val_acc_list, 
                    'val_epoch_list': val_epoch_list,
                    'val_f1_list': val_f1_list,
                    'val_auc_pr_list': val_auc_pr_list,
                    'val_recall_list': val_recall_list
                }
                torch.save(save_dict, save_name)
                
    except Exception as exc:
        print(exc)
        import traceback
        traceback.print_exc()
        
    finally:
        print(f'训练停止于epoch {e}')
    
    # 计算总训练时间
    train_time = time.time() - train_st
    print(f"训练时间: {train_time}")
    
    # 返回训练结果
    return {
        'loss_list': loss_list,
        'acc_list': acc_list,
        'val_acc_list': val_acc_list,
        'val_epoch_list': val_epoch_list,
        'val_f1_list': val_f1_list,
        'val_auc_pr_list': val_auc_pr_list,
        'val_recall_list': val_recall_list,
        'train_time': train_time
    }

def evaluate_model(model, data_loader, device):
    """
    评估模型性能
    
    参数:
        model: 训练好的模型
        data_loader: 数据加载器
        device: 计算设备
    
    返回:
        评估结果
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, target in tqdm(data_loader, desc="评估中"):
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    
    # 计算评估指标
    accuracy = accuracy_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds, average='binary')
    
    # 生成分类报告
    report = classification_report(all_targets, all_preds, target_names=['Negative', 'Positive'])
    
    return {
        'accuracy': accuracy,
        'recall': recall,
        'report': report,
        'predictions': all_preds,
        'targets': all_targets
    }

In [ ]:
def get_best_model(metrics_list, epoch_list, save_path, metric='acc', del_others=True):
    """
    通过指定评估指标找到最佳模型
    
    参数:
        metrics_list: 指标列表（如准确率、F1或AUC-PR）
        epoch_list: 对应的epoch列表
        save_path: 模型保存路径
        metric: 要使用的指标，默认为'acc'，可选'f1'或'auc_pr'
        del_others: 是否删除其他模型
    
    返回:
        best_model_path: 最佳模型路径
    """
    metrics_list = np.array(metrics_list)
    epoch_list = np.array(epoch_list)
    best_index = np.argwhere(metrics_list == np.max(metrics_list))[-1].item()
    best_epoch = epoch_list[best_index]
    best_metric = metrics_list[best_index]
    
    # 根据使用的指标查找对应模型文件
    if metric == 'f1':
        pattern = f"epoch_{best_epoch}_*_f1_{best_metric:.4f}_*.pth"
    elif metric == 'auc_pr':
        pattern = f"epoch_{best_epoch}_*_aucpr_{best_metric:.4f}.pth"
    else:  # 默认使用acc
        pattern = f"epoch_{best_epoch}_acc_{best_metric:.4f}_*.pth"
    
    matching_files = glob.glob(os.path.join(save_path, pattern))
    if not matching_files:
        # 备用搜索方式
        all_model_files = glob.glob(os.path.join(save_path, "*.pth"))
        for file in all_model_files:
            if f"epoch_{best_epoch}_" in file:
                matching_files.append(file)
    
    if not matching_files:
        raise FileNotFoundError(f"找不到对应的模型文件: {pattern}")
    
    best_model_path = matching_files[0]
    print(f"最佳模型 ({metric}={best_metric:.4f}): {os.path.basename(best_model_path)}")
    
    # 删除其他模型
    if del_others:
        for f in os.listdir(save_path):
            if f.endswith('.pth') and os.path.join(save_path, f) != best_model_path:
                os.remove(os.path.join(save_path, f))
    
    return best_model_path

In [ ]:
# 选择要训练的标签ID
LABEL_ID = 1  # 可以根据需要修改

# 获取该标签的数据集
dataset_dict = get_dataset_for_label(
    LABEL_ID, 
    all_data_dict, 
    train_valid_test_ratio=[TRAIN_RATE, (1-TRAIN_RATE)/2, (1-TRAIN_RATE)/2],
    apply_pca_flag=APPLY_PCA, 
    n_components=N_PCA, 
    norm=NORM
)

# 创建训练、测试和验证数据集
train_dataset = BrainVoxelDataset(dataset_dict['train_samples'], dataset_dict['train_labels'])
test_dataset = BrainVoxelDataset(dataset_dict['test_samples'], dataset_dict['test_labels'])
val_dataset = BrainVoxelDataset(dataset_dict['val_samples'], dataset_dict['val_labels'])


# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 设置计算设备
device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 创建模型
feature_dim = dataset_dict['feature_dim']
model = BrainVoxelKAN(feature_dim, 64, NUM_CLASS, FIXED_GRID).to(device)

# 打印模型结构
summary(model)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# 训练模型
training_results = train_brain_voxel_kan(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=EPOCH,
    val_epoch=VAL_EPOCH,
    save_path=SAVE_PATH
)

# 获取最佳模型并评估
best_model_path = get_best_model(
    training_results['val_auc_pr_list'],  # 使用AUC-PR列表
    training_results['val_epoch_list'],
    SAVE_PATH,
    metric='auc_pr'  # 明确指定使用AUC-PR作为指标
)

# 加载最佳模型
best_model = BrainVoxelKAN(feature_dim, 64, NUM_CLASS, FIXED_GRID).to(device)
best_model.load_state_dict(torch.load(best_model_path)['state_dict'])

# 在验证集上评估
print("在验证集上评估最佳模型...")
validation_results = evaluate_model(best_model, val_loader, device)
print(f"验证集准确率: {validation_results['accuracy']}")
print(f"验证集召回率: {validation_results['recall']}")
print("\n分类报告:")
print(validation_results['report'])

# 保存训练过程图表
plt.figure(figsize=(15, 5))

# 绘制损失曲线
plt.subplot(1, 3, 1)
plt.plot(range(len(training_results['loss_list'])), training_results['loss_list'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

# 绘制准确率曲线
plt.subplot(1, 3, 2)
plt.plot(range(len(training_results['acc_list'])), training_results['acc_list'], label='Train Acc')
plt.plot(training_results['val_epoch_list'], training_results['val_acc_list'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 绘制AUC-PR曲线
plt.subplot(1, 3, 3)
plt.plot(training_results['val_epoch_list'], training_results['val_auc_pr_list'], 'g-', label='AUC-PR')
plt.plot(training_results['val_epoch_list'], training_results['val_f1_list'], 'r--', label='F1 Score')
plt.title('AUC-PR & F1 Score')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, f'training_curves_with_auc_pr_label_{LABEL_ID}.png'))
plt.show()

# 保存验证结果
validation_report = f"""
# 验证报告 - 标签 {LABEL_ID}

## 训练信息
- 训练时间: {training_results['train_time']:.2f} 秒
- 总训练轮数: {len(training_results['loss_list'])}
- 最佳模型: {os.path.basename(best_model_path)}
- 学习率: {LR}
- 批量大小: {BATCH_SIZE}
- 固定网格大小: {FIXED_GRID}

## 性能指标
- 验证集准确率: {validation_results['accuracy']:.4f}
- 验证集召回率: {validation_results['recall']:.4f}

## 分类报告
{validation_results['report']}
"""

with open(os.path.join(SAVE_PATH, f'validation_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(validation_report)

print(f"验证报告已保存至: {os.path.join(SAVE_PATH, f'validation_report_label_{LABEL_ID}.txt')}")

In [ ]:
def predict_whole_dataset(model, label_id, all_data_dict, device, apply_pca_flag=True, n_components=15, norm=True, batch_size=64):
 
    """
    使用训练好的模型对全部数据集进行预测
    
    参数:
        model: 训练好的模型
        label_id: 标签ID
        all_data_dict: 数据集字典
        device: 计算设备
        apply_pca_flag: 是否应用PCA
        n_components: PCA保留的主成分数量
        norm: 是否标准化数据
        batch_size: 批处理大小
    
    返回:
        预测结果
    """
    # 获取完整标签集
    train_label_dir = all_data_dict['train_label_dir']
    val_label_dir = all_data_dict['val_label_dir']
    
    # 使用与get_dataset_for_label相同的方法获取数据
    def get_label_file_path(label_id, is_validation=False):
        if is_validation:
            pattern = os.path.join(val_label_dir, f"label_{label_id}_count_*_voxels.npy")
        else:
            pattern = os.path.join(train_label_dir, f"label_{label_id}_count_*_voxels.npy")
        
        matches = glob.glob(pattern)
        return matches[0] if matches else None
    
    # 收集所有数据
    all_samples = []
    all_labels = []
    
    # 获取目标标签的数据（正样本）
    train_file = get_label_file_path(label_id, False)
    val_file = get_label_file_path(label_id, True)
    
    if train_file:
        train_samples = np.load(train_file)
        train_labels = np.ones(len(train_samples))
        all_samples.append(train_samples)
        all_labels.append(train_labels)
    
    if val_file:
        val_samples = np.load(val_file)
        val_labels = np.ones(len(val_samples))
        all_samples.append(val_samples)
        all_labels.append(val_labels)
    
    # 获取其他标签的数据（负样本）
    valid_labels = all_data_dict['valid_labels']
    other_labels = [l for l in valid_labels if l != label_id]
    
    for other_label in other_labels:
        other_train_file = get_label_file_path(other_label, False)
        other_val_file = get_label_file_path(other_label, True)
        
        if other_train_file:
            other_train_samples = np.load(other_train_file)
            other_train_labels = np.zeros(len(other_train_samples))
            all_samples.append(other_train_samples)
            all_labels.append(other_train_labels)
        
        if other_val_file:
            other_val_samples = np.load(other_val_file)
            other_val_labels = np.zeros(len(other_val_samples))
            all_samples.append(other_val_samples)
            all_labels.append(other_val_labels)
    
    # 合并所有数据
    all_samples = np.vstack(all_samples)
    all_labels = np.concatenate(all_labels)
    
    # 应用PCA处理（如果启用）
    if apply_pca_flag:
        if pca_model is not None:
            # 使用训练时保存的PCA模型
            print(f"使用训练时的PCA模型转换预测数据...")
            all_samples_transformed = pca_model.transform(all_samples)
            if norm:
                # 应用相同的标准化
                all_samples_transformed = (all_samples_transformed - np.min(all_samples_transformed, axis=0)) / (np.max(all_samples_transformed, axis=0) - np.min(all_samples_transformed, axis=0) + 1e-10)
            all_samples = all_samples_transformed
        else:
            # 重新计算PCA（不推荐，但作为备选）
            print(f"警告：未提供训练时的PCA模型，重新计算PCA可能导致结果不一致")
            all_samples, _ = apply_pca(all_samples, n_components, norm)
    
    # 创建数据集和加载器
    full_dataset = BrainVoxelDataset(all_samples, all_labels)
    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)
    
    # 进行预测
    model.eval()
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for data, _ in tqdm(full_loader, desc="全数据集预测"):
            data = data.to(device)
            output = model(data)
            probs = torch.softmax(output, dim=1)
            _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # 保存正类的概率
    
    # 计算评估指标
    accuracy = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds, average='binary')
    
    # 计算AUC-PR（精确率-召回率曲线下面积）
    precision, recall_curve_points, _ = precision_recall_curve(all_labels, all_probs)
    auc_pr = auc(recall_curve_points, precision)
    
    # 生成分类报告
    report = classification_report(all_labels, all_preds, target_names=['Negative', 'Positive'])
    
    return {
        'accuracy': accuracy,
        'recall': recall,
        'auc_pr': auc_pr,
        'report': report,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels,
        'samples': all_samples
    }
    
# 可视化精确率-召回率曲线
def plot_precision_recall_curve(labels, probabilities, save_path=None):
    """
    绘制精确率-召回率曲线
    
    参数:
        labels: 真实标签
        probabilities: 预测为正类的概率
        save_path: 保存路径，None表示不保存
    """
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    auc_pr = auc(recall, precision)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, lw=2, label=f'PR Curve (AUC = {auc_pr:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc='lower left')
    plt.grid(True)
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()

In [ ]:
# 在全数据集上应用最佳模型
print("在全数据集上进行预测...")
full_prediction_results = predict_whole_dataset(
    best_model, 
    LABEL_ID, 
    all_data_dict, 
    device,
    apply_pca_flag=APPLY_PCA,  # 确保这些参数与训练时相同
    n_components=N_PCA,
    norm=NORM,
    batch_size=BATCH_SIZE
)

# 打印性能指标
print(f"全数据集准确率: {full_prediction_results['accuracy']:.4f}")
print(f"全数据集召回率: {full_prediction_results['recall']:.4f}")
print(f"全数据集AUC-PR: {full_prediction_results['auc_pr']:.4f}")
print("\n全数据集分类报告:")
print(full_prediction_results['report'])

# 绘制精确率-召回率曲线
plot_precision_recall_curve(
    full_prediction_results['labels'],
    full_prediction_results['probabilities'],
    os.path.join(SAVE_PATH, f'pr_curve_label_{LABEL_ID}.png')
)

# 保存全数据集评估结果
full_dataset_report = f"""
# 全数据集评估报告 - 标签 {LABEL_ID}

## 性能指标
- 准确率: {full_prediction_results['accuracy']:.4f}
- 召回率: {full_prediction_results['recall']:.4f}
- AUC-PR: {full_prediction_results['auc_pr']:.4f}
- 正样本数量: {np.sum(full_prediction_results['labels'] == 1)}
- 负样本数量: {np.sum(full_prediction_results['labels'] == 0)}
- 预测为正的样本数: {np.sum(full_prediction_results['predictions'] == 1)}
- 预测为负的样本数: {np.sum(full_prediction_results['predictions'] == 0)}

## 分类报告
{full_prediction_results['report']}
"""

with open(os.path.join(SAVE_PATH, f'full_dataset_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(full_dataset_report)

print(f"全数据集评估报告已保存至: {os.path.join(SAVE_PATH, f'full_dataset_report_label_{LABEL_ID}.txt')}")

In [ ]:
# 在验证集上进行全面评估
print("在验证集上进行全面评估...")
model.eval()
all_preds = []
all_probs = []
all_targets = []

# 收集验证集上的预测结果和概率
with torch.no_grad():
    for data, target in tqdm(val_loader, desc="验证集评估"):
        data, target = data.to(device), target.to(device)
        output = model(data)
        probs = torch.softmax(output, dim=1)
        _, preds = torch.max(output, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # 保存正类的概率
        all_targets.extend(target.cpu().numpy())

# 计算各种评估指标
accuracy = accuracy_score(all_targets, all_preds)
recall = recall_score(all_targets, all_preds, average='binary')
precision, recall_points, _ = precision_recall_curve(all_targets, all_probs)
auc_pr = auc(recall_points, precision)
report = classification_report(all_targets, all_preds, target_names=['Negative', 'Positive'])
conf_matrix = confusion_matrix(all_targets, all_preds)

# 打印主要评估指标
print(f"验证集准确率: {accuracy:.4f}")
print(f"验证集召回率: {recall:.4f}")
print(f"验证集AUC-PR: {auc_pr:.4f}")
print(f"验证集正样本数: {sum(all_targets)}")
print(f"验证集负样本数: {len(all_targets) - sum(all_targets)}")
print(f"预测为正的样本数: {sum(all_preds)}")
print(f"预测为负的样本数: {len(all_preds) - sum(all_preds)}")
print("\n分类报告:")
print(report)
print("\n混淆矩阵:")
print(conf_matrix)

# 绘制PR曲线
plt.figure(figsize=(10, 8))
plt.subplot(2, 2, 1)
precision, recall_points, _ = precision_recall_curve(all_targets, all_probs)
plt.plot(recall_points, precision, lw=2, label=f'PR Curve (AUC = {auc_pr:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)

# 绘制ROC曲线
plt.subplot(2, 2, 2)
fpr, tpr, _ = roc_curve(all_targets, all_probs)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)

# 绘制概率分布
plt.subplot(2, 2, 3)
plt.hist([all_probs[i] for i in range(len(all_targets)) if all_targets[i] == 1], 
         bins=20, alpha=0.5, label='Positive Samples')
plt.hist([all_probs[i] for i in range(len(all_targets)) if all_targets[i] == 0], 
         bins=20, alpha=0.5, label='Negative Samples')
plt.xlabel('Prediction Probability')
plt.ylabel('Sample Count')
plt.title('Probability Distribution')
plt.legend()
plt.grid(True)

# 绘制混淆矩阵
plt.subplot(2, 2, 4)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, f'validation_performance_label_{LABEL_ID}.png'))
plt.show()

# 保存验证集评估报告
validation_report_complete = f"""
# 验证集全面评估报告 - 标签 {LABEL_ID}

## 性能指标
- 准确率: {accuracy:.4f}
- 精确率: {precision_score(all_targets, all_preds):.4f}
- 召回率: {recall:.4f}
- F1分数: {f1_score(all_targets, all_preds):.4f}
- AUC-PR: {auc_pr:.4f}
- ROC-AUC: {roc_auc:.4f}

## 样本分布
- 总样本数: {len(all_targets)}
- 正样本数: {sum(all_targets)} ({sum(all_targets)/len(all_targets)*100:.2f}%)
- 负样本数: {len(all_targets) - sum(all_targets)} ({(len(all_targets) - sum(all_targets))/len(all_targets)*100:.2f}%)
- 预测为正的样本数: {sum(all_preds)} ({sum(all_preds)/len(all_preds)*100:.2f}%)
- 预测为负的样本数: {len(all_preds) - sum(all_preds)} ({(len(all_preds) - sum(all_preds))/len(all_preds)*100:.2f}%)

## 混淆矩阵
- 真正例(TP): {conf_matrix[1][1]}
- 假正例(FP): {conf_matrix[0][1]}
- 真负例(TN): {conf_matrix[0][0]}
- 假负例(FN): {conf_matrix[1][0]}

## 分类报告
{report}

## 阈值分析
以下是不同预测概率阈值下的性能：

| 阈值 | 精确率 | 召回率 | F1分数 |
|------|--------|--------|--------|
"""

# 添加不同阈值下的性能
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for threshold in thresholds:
    threshold_preds = [1 if prob >= threshold else 0 for prob in all_probs]
    threshold_precision = precision_score(all_targets, threshold_preds, zero_division=0)
    threshold_recall = recall_score(all_targets, threshold_preds, zero_division=0)
    threshold_f1 = f1_score(all_targets, threshold_preds, zero_division=0)
    validation_report_complete += f"| {threshold:.1f} | {threshold_precision:.4f} | {threshold_recall:.4f} | {threshold_f1:.4f} |\n"

with open(os.path.join(SAVE_PATH, f'validation_complete_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(validation_report_complete)

print(f"验证集全面评估报告已保存至: {os.path.join(SAVE_PATH, f'validation_complete_report_label_{LABEL_ID}.txt')}")

# 额外添加特征重要性分析
feature_importance = analyze_kan_model(
    best_model,
    dataset_dict['val_samples'],
    os.path.join(SAVE_PATH, f'feature_importance_validation_label_{LABEL_ID}.png'),
    apply_pca_flag=APPLY_PCA 
)

# 保存特征重要性数据
np.save(os.path.join(SAVE_PATH, f'feature_importance_validation_label_{LABEL_ID}.npy'), feature_importance)

In [ ]:
def analyze_kan_model(model, data_samples, save_path=None, apply_pca_flag=True):
    """
    简单分析KAN模型的特征重要性
    
    参数:
        model: 训练好的KAN模型
        data_samples: 数据样本
        save_path: 保存路径，None表示不保存
        apply_pca_flag: 是否应用了PCA，影响特征命名
    """
    # 获取模型输入层的权重
    input_weights = model.kan.layers[0].base_linear.weight.data.cpu().numpy()
    
    # 计算特征的平均绝对权重值（简单的重要性度量）
    feature_importance = np.mean(np.abs(input_weights), axis=0)
    
    # 找出前20个最重要的特征
    top_n = min(20, len(feature_importance))
    top_indices = np.argsort(feature_importance)[-top_n:][::-1]
    top_importance = feature_importance[top_indices]
    
    # 可视化特征重要性
    plt.figure(figsize=(12, 8))
    
    feature_names = [f"{'PC' if apply_pca_flag else 'Feature'} {i+1}" for i in top_indices]
    
    plt.barh(range(top_n), top_importance, align='center')
    plt.yticks(range(top_n), feature_names)
    plt.xlabel('Mean Absolute Weight')
    plt.title(f"Top {top_n} {'Principal Component' if apply_pca_flag else 'Feature'} Importance")
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()
    
    # 保存特征重要性数据
    importance_data = {
        'feature_index': np.arange(len(feature_importance)),
        'importance': feature_importance,
        'is_pca': apply_pca_flag
    }
    
    return importance_data

# 分析模型
# 分析模型
print("分析模型特征重要性...")
importance_data = analyze_kan_model(
    best_model,
    full_prediction_results['samples'],
    os.path.join(SAVE_PATH, f'feature_importance_label_{LABEL_ID}.png'),
    apply_pca_flag=APPLY_PCA
)

# 保存特征重要性数据
np.save(os.path.join(SAVE_PATH, f'feature_importance_label_{LABEL_ID}.npy'), importance_data)

In [ ]:
# 总结训练和评估结果
summary_report = f"""
# 模型训练及评估总结 - 标签 {LABEL_ID}

## 模型和训练配置
- 模型: BrainVoxel_1DKAN
- 特征处理: {"PCA降维" if APPLY_PCA else "原始特征"}
- 特征维度: {feature_dim}{"（PCA降维后）" if APPLY_PCA else ""}
- 隐藏层维度: 64
- 输出类别数: {NUM_CLASS}
- 固定网格大小: {FIXED_GRID}
- 批量大小: {BATCH_SIZE}
- 学习率: {LR}
- 权重衰减: {WEIGHT_DECAY}
- 训练轮数: {len(training_results['loss_list'])}
- 训练时间: {training_results['train_time']:.2f} 秒

## 训练集分割
- 原训练集分割比例: {TRAIN_RATE}:{TEST_RATE}
- 训练集样本数: {len(train_dataset)}
- 测试集样本数: {len(test_dataset)}
- 验证集样本数: {len(val_dataset)}
- 正类样本比例: 约 50%

## 性能指标
- 测试集最佳准确率: {max(training_results['val_acc_list']):.4f}
- 验证集准确率: {validation_results['accuracy']:.4f}
- 验证集召回率: {validation_results['recall']:.4f}
- 全数据集准确率: {full_prediction_results['accuracy']:.4f}
- 全数据集AUC-PR: {full_prediction_results['auc_pr']:.4f}

## 重要发现
- KAN模型在脑体素分类任务上表现良好
- 简化的训练流程提高了训练效率
- 固定网格大小简化了模型，同时保持了分类性能
- 特征重要性分析可以帮助理解模型决策

## 建议
- 可以尝试不同的网格大小以平衡性能和复杂度
- 考虑对最重要的特征进行进一步分析
- 将训练流程应用于其他标签
"""

with open(os.path.join(SAVE_PATH, f'summary_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(summary_report)

print(f"训练和评估总结已保存至: {os.path.join(SAVE_PATH, f'summary_report_label_{LABEL_ID}.txt')}")
print("完成！")